# param-group-dict-list — ex1: build differential-LR param groups for encoder vs head

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `param-group-dict-list`. Running the final beacon cell reports progress against the `Config: param-group dict list` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: param-group dict list` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`param-group-dict-list`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "param-group-dict-list"
DD_SUBTOPIC = "Config: param-group dict list"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Config: param-group dict list — quick refresher

PyTorch's optimizer constructor accepts EITHER a flat parameter iterable OR a list of dicts. The dict form lets you set different hyperparameters per group (different LR for backbone vs head, no weight decay on biases, etc.):

```python
optimizer = torch.optim.Adam([
    {'params': encoder.parameters(), 'lr': 1e-4},
    {'params': head.parameters(),    'lr': 1e-2},
], lr=1e-3)  # ← default LR (used when a group omits the key)
```

**Each dict MUST have a `params` key.** Other keys (`lr`, `weight_decay`, `momentum`, `betas`) override the constructor defaults for THAT group only.

**Keys you don't set fall through to the top-level default.** If a group dict has `{'params': ..., 'lr': 1e-4}` and you pass `weight_decay=0.01` at the constructor level, that group gets `weight_decay=0.01`. The optimizer fills in missing keys.

**Common pattern — no decay on biases/LayerNorm.** Two groups, same LR, different `weight_decay`. The split is structural, not by optimizer choice.

**Calling `.parameters()` returns a GENERATOR**, which is single-use. If you build two groups from the same module you'll exhaust the iterator. Wrap with `list(model.parameters())` if you need to introspect or reuse.

### Exercise 1 — build differential-LR param groups for encoder vs head

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `[{'params': ..., 'lr': ...}, ...]` dict-list construction to give different learning rates to two halves of a model, and verify the optimizer reads back each group's LR correctly.
> Keywords: param-groups, differential-lr, fine-tuning, transfer
> ```

**KCs targeted:** `param-group-dict-with-lr-override`, `list-vs-generator-parameters`

Implement `ex1_make_param_groups(encoder, head, encoder_lr, head_lr)`. The classical transfer-learning differential-LR setup.

1. Build a list of TWO dicts:
   - `{'params': list(encoder.parameters()), 'lr': encoder_lr}`
   - `{'params': list(head.parameters()), 'lr': head_lr}`
2. Return the list.

Why wrap in `list(...)`: `.parameters()` returns a single-use generator. If the optimizer iterates it more than once (it doesn't normally, but downstream tooling like checkpoint loaders sometimes does), the second pass yields nothing. Materializing into a list is the safe default.

Inputs:
- `encoder`, `head`: `nn.Module` instances.
- `encoder_lr`, `head_lr`: floats.

Output: `list[dict]` ready to pass as the first argument to any `torch.optim` optimizer.

In [ ]:
def ex1_make_param_groups(encoder, head, encoder_lr: float, head_lr: float):
    """Two-group param dict-list with per-group lr."""
    raise NotImplementedError()


def _test_ex1():
    import torch.nn as nn

    encoder = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 16))
    head = nn.Linear(16, 3)

    groups = ex1_make_param_groups(encoder, head, encoder_lr=1e-4, head_lr=1e-2)
    assert isinstance(groups, list), f'must return a list, got {type(groups).__name__}'
    assert len(groups) == 2, f'must return 2 groups, got {len(groups)}'

    # === Each group is a dict with the right keys ===
    for i, g in enumerate(groups):
        assert isinstance(g, dict), f'group {i} must be a dict, got {type(g).__name__}'
        assert 'params' in g, f'group {i} missing params key'
        assert 'lr' in g, f'group {i} missing lr key'

    # === Group order: encoder first, head second ===
    assert groups[0]['lr'] == 1e-4, f'group 0 lr should be encoder_lr=1e-4, got {groups[0]["lr"]}'
    assert groups[1]['lr'] == 1e-2, f'group 1 lr should be head_lr=1e-2, got {groups[1]["lr"]}'

    # === Param counts match ===
    enc_params = list(encoder.parameters())
    head_params = list(head.parameters())
    assert len(groups[0]['params']) == len(enc_params), (
        f'encoder group has {len(groups[0]["params"])} params, expected {len(enc_params)}'
    )
    assert len(groups[1]['params']) == len(head_params), (
        f'head group has {len(groups[1]["params"])} params, expected {len(head_params)}'
    )
    # Encoder has 4 tensors (2 linears × {weight,bias}), head has 2.
    assert len(groups[0]['params']) == 4, f'expected 4 encoder params, got {len(groups[0]["params"])}'
    assert len(groups[1]['params']) == 2, f'expected 2 head params, got {len(groups[1]["params"])}'

    # === params is a LIST (materialized), not a generator ===
    import types
    for i, g in enumerate(groups):
        assert not isinstance(g['params'], types.GeneratorType), (
            f'group {i} params is a generator — should be materialized to a list'
        )

    # === Plug into a real optimizer; read back per-group LR ===
    opt = t.optim.Adam(groups)
    assert len(opt.param_groups) == 2
    assert opt.param_groups[0]['lr'] == 1e-4
    assert opt.param_groups[1]['lr'] == 1e-2
    # Other defaults filled in by Adam (e.g. betas).
    for g in opt.param_groups:
        assert 'betas' in g, 'Adam should fill in betas default per group'

    # === Run one step; encoder moves slower than head ===
    x = t.randn(4, 8)
    logits = head(encoder(x))
    loss = logits.pow(2).mean()
    # Snapshot a representative param from each group.
    enc_w = encoder[0].weight.detach().clone()
    head_w = head.weight.detach().clone()
    opt.zero_grad()
    loss.backward()
    opt.step()
    enc_delta = (encoder[0].weight - enc_w).abs().mean().item()
    head_delta = (head.weight - head_w).abs().mean().item()
    # Both moved, but per-step magnitude is gated by LR.
    # (Not a strict assertion — gradient magnitudes differ — but encoder LR is 100x smaller.)
    assert enc_delta > 0, 'encoder weights should have moved at all'
    assert head_delta > 0, 'head weights should have moved at all'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_make_param_groups(encoder, head, encoder_lr, head_lr):
    return [
        {'params': list(encoder.parameters()), 'lr': encoder_lr},
        {'params': list(head.parameters()),    'lr': head_lr},
    ]
```

**Why both groups need `'params'`.** It's the ONE required key in the param-group spec. Forgetting it raises `ValueError: optimizer got an empty parameter list` or worse, silently constructs a no-param group.

**What if I want the same LR for both halves?** Drop the `'lr'` key — the optimizer's top-level `lr=` kwarg falls through. Param groups are most useful when DIFFERENT hparams apply, not for cosmetic grouping.

**Three-group variant — no-decay biases.** A common extension is to split `params` into two groups, one for `weight` tensors (with `weight_decay=0.01`) and one for biases + LayerNorm (with `weight_decay=0`). Same mechanism — just three dicts in the list, filtered by `p.dim() > 1` and name.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()